In [6]:
import pandas as pd
from pandasql import sqldf
import datetime
import numpy as np
import warnings
from arch import arch_model
import requests
from io import BytesIO
warnings.filterwarnings("ignore")

file_url = 'https://www.iif.com/Portals/0/Files/Databases/monthly_em_portfolio_flows_database.xlsx?ver=2025-03-13-063222-833'
response = requests.get(file_url)

if response.headers.get('Content-Type') == 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet':
    excel_data = BytesIO(response.content)
    df = pd.read_excel(excel_data, sheet_name='Country Flows_nw', header=3)
else:
    print("Failed to download a valid Excel file. The URL might not be a direct link.")

equity = df[df.columns[[0,1,3,5,7,9,11,13,14,16,45,60]]]
equity.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
equity['Date'].fillna(0, inplace=True)
date_new = [x for x in equity['Date'] if isinstance(x, datetime.date)]
equity = equity[equity['Date'].isin(date_new)]

debt = df[df.columns[[0,2,4,6,8,10,12,15,61]]]
debt.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
debt['Date'].fillna(0, inplace=True)
date_new = [x for x in debt['Date'] if isinstance(x, datetime.date)]
debt = debt[debt['Date'].isin(date_new)]
debt.rename(columns={k:v for (k,v) in zip(debt.columns, equity.columns.drop(['Taiwan, China', 'Vietnam', 'Sri lanka']))}, inplace=True)

equity.set_index('Date', inplace=True)
debt.set_index('Date', inplace=True)

df_cf = pd.DataFrame(index=debt.index)

df_cf['Portfolio Debt'] = debt.sum(axis=1)/1000
df_cf['Portfolio Equity'] = equity.sum(axis=1)/1000
df_cf.reset_index(inplace=True)

df_cf

,Date,Portfolio Debt,Portfolio Equity
0,2000-01-15,-0.3207,3.27702
1,2000-02-15,0.6629,3.49665
2,2000-03-15,1.4968,5.73722
3,2000-04-15,0.6739,0.01669
4,2000-05-15,-1.6931,1.10859
...,...,...,...
309,2025-10-15,-16.370654,-6.386478
310,2025-11-15,18.892516,-34.397636
311,2025-12-15,-3.544351,-1.914915
312,2026-01-15,10.7273,18.523097


In [7]:
from datetime import datetime

with pd.ExcelWriter(r"C:\Users\AhmadAizudeen\The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\SEACEN Faculty & Research - RSU\00_Datasets\Economics Indicator Database\capital_flow.xlsx") as writer:        
    # use to_excel function and specify the sheet_name and index 
    # to store the dataframe in specified sheet
    equity.to_excel(writer, sheet_name="equity")
    debt.to_excel(writer, sheet_name="debt")
    df_cf.to_excel(writer, sheet_name="df_cf")


In [8]:
import pandas as pd

df = pd.read_excel(
    r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\download zip\VIX_History.xlsx",
    sheet_name=1,
)
df["Date"] = df["Month"].astype(str) + "-" + df["Year"].astype(str)
df["Date"] = pd.to_datetime(df["Date"]) + pd.offsets.MonthEnd(0)
df = df.rename(columns={"avg(CLOSE)": "CLOSE"})
df = df[["Date", "CLOSE"]]

with pd.ExcelWriter("vix index.xlsx") as writer:        
    # use to_excel function and specify the sheet_name and index 
    # to store the dataframe in specified sheet
    df.to_excel(writer, sheet_name="vix index")

In [ ]:
import xlwings as xw
import plotly.express as px
from data import capital_flows
from datetime import date
today = date.today()

formatted_date = today.strftime("%Y-%m-%d")
# Create workbook
wb = xw.Book(r"C:\Users\AhmadAizudeen\The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\SEACEN Faculty & Research - RSU\00_Datasets\RSU Macrofinancial Indicators\AA\Portfolio Flows.xlsx")

# Add sheets for figures
wb.sheets[0].name = 'Figure 1'
wb.sheets.add(name='Figure 2')
wb.sheets.add(name='Figure 3')

# Add sheets for data
wb.sheets.add(name='Currency Amount Outstanding')
wb.sheets.add(name='Bank Amount Outstanding')
wb.sheets.add(name='Claim Amount Outstanding')

# Generate data
df1 = currency_amount_outstanding()
print('Generate data for currency amount outstanding')
df2 = bank_amount_outstanding()
print('Generate data for bank amount outstanding')
df3 = claim_amount_outstanding()
print('Generate data for claim amount outstanding')

# Generate figures
fig1 = simple_fig(df3)
fig2 = fig_amount_outstanding(df1, 'Currency')
fig3 = fig_amount_outstanding(df2, 'Counterparty sector')

# Insert figures
wb.sheets['Figure 1'].pictures.add(fig1, name='fig1', update=True)
wb.sheets['Figure 2'].pictures.add(fig2, name='fig2', update=True)
wb.sheets['Figure 3'].pictures.add(fig3, name='fig3', update=True)

df3 = df3.pivot(columns='Period', index='Reporting country', values='Value').reset_index()
df2 = df2.pivot(columns='Year', index=['Balance sheet position', 'Counterparty sector'], values='Value').reset_index()
df1 = df1.pivot(columns='Year', index=['Balance sheet position', 'Currency'], values='Value').reset_index()


# Insert DataFrames
wb.sheets['Currency Amount Outstanding']["A1"].options(index=False).value = df1
wb.sheets['Bank Amount Outstanding']["A1"].options(index=False).value = df2
wb.sheets['Claim Amount Outstanding']["A1"].options(index=False).value = df3

# Save workbook
wb.save(r"C:\Users\AhmadAizudeen\The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\SEACEN Faculty & Research - RSU\02_Requests\01 CY common charts and figures" + f"\({formatted_date}) CY data and figures.xlsx")
wb.close()

In [ ]:
AhmadAizudeen = {'email': 'aizudeenjapri@gmail.com',
                'interest': ['data science and analytics', 'programming', 'machine learning', 'data engineering', 'Artificial Intelligence'],
                'skills': {'Python': ['pandas', 'pandasql', 'shiny', 'scikit-learn', 'django', 'tensorflow', 'plotly'],
                           'SQL': ['PostgreSQL', 'MSSQL', 'SQLite'],
                           'Data Visualization' : ['Power BI', 'Tableau', 'Excel']},
                'open for work': True,
                'criminal records': None,
                }

AhmadAizudeen

{'email': 'aizudeenjapri@gmail.com',
 'interest': ['data science and analytics',
  'programming',
  'machine learning',
  'data engineering',
  'Artificial Intelligence'],
 'skills': {'Python': ['pandas',
   'pandasql',
   'shiny',
   'scikit-learn',
   'django',
   'tensorflow',
   'plotly'],
  'SQL': ['PostgreSQL', 'MSSQL', 'SQLite'],
  'Data Visualization': ['Power BI', 'Tableau']},
 'open for work': True,
 'criminal records': None}

In [32]:
import pandas as pd
from datetime import datetime
import os
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings("ignore")

year_use = 2019
df_init = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\data.xlsx", sheet_name='BOP')

path_dict = dict()
list_1 = ['CB BOP Quarterly.xlsx']
# directory_path= r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
directory_path= r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
for filename in os.listdir(directory_path):
    if filename in ['01 Emerging Market', '02 G7 Countries']:
        continue
    file_path = os.path.join(directory_path, filename)
    list_temp = []
    for filename2 in os.listdir(file_path):
        if any(word in str(filename2) for word in list_1):            
            list_temp.append(filename2)
            path_dict[filename] = list_temp
        else:
            continue

df_path = []
for k, v in path_dict.items():
    df_path.append(os.path.join(directory_path, k, v[0]))
    print(v[0]) ## to check the file name

df_all_annual = pd.DataFrame()
for x, y in enumerate(df_path):
    df = pd.read_excel(y)
    df = df[df[df.columns[0]].isin(df_init['Item'])]
    df_all_annual = pd.concat([df_all_annual, df]) 

df_all_annual = df_all_annual[df_all_annual['Region'].isin(df_init['Country'])]
# df_all_annual
df_all_annual.rename(columns={df_all_annual.columns[0]: 'Type'}, inplace=True)

df_all_annual_2 = df_all_annual[df_all_annual.columns[:4].to_list() + [dt for dt in df_all_annual.columns[4:].sort_values(ascending=True).to_list() if dt >= datetime(year_use, 1, 1)]]

seacen_country = ['Papua New Guinea', 'Vietnam', 'Nepal', 'India', 'Indonesia', 'Laos', 'Sri Lanka', 'Hong Kong SAR (China)', 'Philippines', 'Taiwan', 'Malaysia', 'Mongolia', 'China', 'Cambodia', 'Thailand', 'Singapore', 'South Korea', 'Brunei', 'Myanmar']

to_replace = {k:v for k,v in zip(df_init['Item'], df_init['short_title'])}

df_all_annual_2.replace(to_replace, inplace=True)

for x in df_all_annual_2['Type'].values:
    var1 = list(set(df_all_annual_2[(df_all_annual_2['Type'] == x)]['Region'].values).symmetric_difference(set(seacen_country)))
    if len(var1) > 0:
        for y in var1:
            df_temp1 = pd.DataFrame({"Type": [x], "Region": [y]})
            df_all_annual_2 = pd.concat([df_all_annual_2, df_temp1], axis=0)
    else:
        continue
df_all_annual_2.replace('South Korea', 'Korea', inplace=True)

# df_all_annual_2 = df_all_annual_2[(df_all_annual_2['Region'].isin(df_init['Country'].unique()) & ~(df_all_annual_2['Unit'].isna()))] #remove this if needed
df_np = df_all_annual_2[df_all_annual_2['Region'] == 'Nepal']
df_all_annual_2 = df_all_annual_2[df_all_annual_2['Region'] != 'Nepal']
dt_ch = [dt for dt in df_all_annual_2.columns[4:].sort_values(ascending=True).to_list() if dt.month in [4,7,10,1]]
new_dt = [dt - relativedelta(months=1) for dt in dt_ch]
df_np = df_np[df_np.columns[:4].to_list() + dt_ch]
df_all_annual_2 = df_all_annual_2[df_all_annual_2.columns[:4].to_list() + new_dt]
df_np.rename(columns={k:v for k,v in zip(dt_ch, new_dt)}, inplace=True)

df_all_annual_2 = pd.concat([df_all_annual_2, df_np], axis=0)
df_all_annual_2.sort_values(by=['Type', 'Region'], inplace=True)
df_all_annual_2.reset_index(drop=True, inplace=True)
df_all_annual_2


KH CB BOP Quarterly.xlsx
CH CB BOP Quarterly.xlsx
TW CB BOP Quarterly.xlsx
HK CB BOP Quarterly.xlsx
IN CB BOP Quarterly.xlsx
ID CB BOP Quarterly.xlsx
LA CB BOP Quarterly.xlsx
MY CB BOP Quarterly.xlsx
MG CB BOP Quarterly.xlsx
MM CB BOP Quarterly.xlsx
NP CB BOP Quarterly.xlsx
SG CB BOP Quarterly.xlsx
LK CB BOP Quarterly.xlsx
TH CB BOP Quarterly.xlsx
VN CB BOP Quarterly.xlsx


,Type,Region,Unit,Last Update Time,2021-12-01 00:00:00,2022-03-01 00:00:00,2022-06-01 00:00:00,2022-09-01 00:00:00,2022-12-01 00:00:00,2023-03-01 00:00:00,2023-06-01 00:00:00,2023-09-01 00:00:00,2023-12-01 00:00:00,2024-03-01 00:00:00,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00
0,Currency and Deposits Assets,Brunei,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Currency and Deposits Assets,Cambodia,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Currency and Deposits Assets,China,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Currency and Deposits Assets,Hong Kong SAR (China),USD mn,2025-09-19,12961.755228,19474.239441,6679.057714,2587.057070,-9920.992074,-7754.824334,-11342.522715,51315.614561,34534.650086,-32138.509539,10339.282973,4845.683366,-15195.05743,25530.31504,1724.540003,NaN
4,Currency and Deposits Assets,India,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
623,Trade Credits and Advances Liabilities,Singapore,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
624,Trade Credits and Advances Liabilities,Sri Lanka,USD mn,2025-12-31,-746.254118,-296.617618,-138.733208,-195.918812,-263.785681,-44.800000,-34.684153,-83.000000,-97.631693,-36.200000,-13.800000,-38.000000,-82.00000,-16.20000,-35.000000,-28.8
625,Trade Credits and Advances Liabilities,Taiwan,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
626,Trade Credits and Advances Liabilities,Thailand,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
df_all_annual_2.to_excel('TEST.xlsx')

In [30]:
# numm = []
# for x in df_all_annual_2['Region'].unique():
#     count = df_all_annual_2[df_all_annual_2['Region'] == x]['Type'].count()
#     numm.append(count)

# print(numm)

# df_all_annual_2['Type'].unique()

df_init

,Country,Country code,Item,group,short_title,desc,desc2
0,Cambodia,KH,BoP: Goods (f1) (f20) (f39),Current Account,Trade Balance (Goods),BoP: CU: GS: Goods,BoP: Current Account: Goods
1,Cambodia,KH,BoP: Services (f4) (f23) (f42),Current Account,Trade Balance (Services),BoP: CU: GS: Services (SV),BoP: Current Account: Services
2,Cambodia,KH,BoP: Primary Income (f7) (f26) (f45),Current Account,Investment Income,BoP: CU: Primary Income (PI),BoP: Current Account: Primary Income
3,Cambodia,KH,BoP: Secondary Income (f10) (f29) (f48),Current Account,Current Transfers,Bop: CU: Secondary Income,BoP: Current Account: Secondary Income
4,Cambodia,KH,BoP: FA: Direct Investment (f16) (f35) (f54),FDI Assets,FDI Assets,BoP: FA: DI: Net Acquisition of Financial Asse...,BoP: Financial Account: Direct Investment: Assets
...,...,...,...,...,...,...,...
169,Laos,LA,BoP: FA: PI: FD: Assets,Financial Derivative Assets,Financial Derivative Assets,BoP: Financial Account: Financial Derivatives ...,NaN
170,Laos,LA,BoP: FA: PI: FD: Liabilities,Financial Derivative Liabilities,Financial Derivative Liabilities,BoP: Financial Account: Financial Derivatives ...,NaN
171,Laos,LA,BoP: FA: OI: Assets,Other Investment Assets,Other Investment Assets,BoP: Financial Account: Other Investment: Assets,NaN
172,Laos,LA,BoP: FA: OI: Liabilities,Other Investment Liabilities,Other Investment Liabilities,BoP: Financial Account: Other Investment: Liab...,NaN


In [15]:
import pandas as pd
from datetime import datetime
import os
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings("ignore")

year_use = 2019
df_init = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\data.xlsx", sheet_name='IIP')
path_dict = dict()
list_1 = ['CB IIP Quarterly.xlsx']
# directory_path= r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
directory_path= r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
for filename in os.listdir(directory_path):
    if filename in ['01 Emerging Market', '02 G7 Countries']:
        continue
    file_path = os.path.join(directory_path, filename)
    list_temp = []
    for filename2 in os.listdir(file_path):
        if any(word in str(filename2) for word in list_1):            
            list_temp.append(filename2)
            path_dict[filename] = list_temp
        else:
            continue

df_path = []
for k, v in path_dict.items():
    df_path.append(os.path.join(directory_path, k, v[0]))
    print(v[0]) ## to check the file name

df_all_annual = pd.DataFrame()
for x, y in enumerate(df_path):
    df = pd.read_excel(y)
    df = df[df[df.columns[0]].isin(df_init['Item'])]
    df_all_annual = pd.concat([df_all_annual, df]) 

df_all_annual = df_all_annual[df_all_annual['Region'].isin(df_init['Country'])]

df_all_annual.rename(columns={df_all_annual.columns[0]: 'Type'}, inplace=True)

df_all_annual_2 = df_all_annual[df_all_annual.columns[:4].to_list() + [dt for dt in df_all_annual.columns[4:].sort_values(ascending=True).to_list() if dt >= datetime(year_use, 1, 1)]]

seacen_country = ['Papua New Guinea', 'Vietnam', 'Nepal', 'India', 'Indonesia', 'Laos', 'Sri Lanka', 'Hong Kong SAR (China)', 'Philippines', 'Taiwan', 'Malaysia', 'Mongolia', 'China', 'Cambodia', 'Thailand', 'Singapore', 'South Korea', 'Brunei', 'Myanmar']

to_replace = {k:v for k,v in zip(df_init['Item'], df_init['short_title'])}

df_all_annual_2.replace(to_replace, inplace=True)

for x in df_all_annual_2['Type'].values:
    var1 = list(set(df_all_annual_2[(df_all_annual_2['Type'] == x)]['Region'].values).symmetric_difference(set(seacen_country)))
    if len(var1) > 0:
        for y in var1:
            df_temp1 = pd.DataFrame({"Type": [x], "Region": [y]})
            df_all_annual_2 = pd.concat([df_all_annual_2, df_temp1], axis=0)
    else:
        continue
df_all_annual_2.replace('South Korea', 'Korea', inplace=True)

# df_all_annual_2 = df_all_annual_2[(df_all_annual_2['Region'].isin(df_init['Country'].unique()) & ~(df_all_annual_2['Unit'].isna()))] #remove this if needed

df_all_annual_2.sort_values(by=['Type', 'Region'], inplace=True)
df_all_annual_2.reset_index(drop=True, inplace=True)
df_all_annual_2


KH CB IIP Quarterly.xlsx
CH CB IIP Quarterly.xlsx
HK CB IIP Quarterly.xlsx
IN CB IIP Quarterly.xlsx
ID CB IIP Quarterly.xlsx
KR CB IIP Quarterly.xlsx
MY CB IIP Quarterly.xlsx
MG CB IIP Quarterly.xlsx
PH CB IIP Quarterly.xlsx
SG CB IIP Quarterly.xlsx
LK CB IIP Quarterly.xlsx
TH CB IIP Quarterly.xlsx


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\AhmadAizudeen\\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\\Roger and Aizudeen\\Country BoP and IIP Data\\Phillipines (PH)\\PH CB IIP Quarterly.xlsx'

In [19]:
df_all_annual_2.to_excel(r"IIP_data_output.xlsx", index=False)

In [12]:
from data_section_2 import iip_quarterly
x1, y1, z1 = iip_quarterly()
y1

KH IMF IIP Quarterly.xlsx
CH IMF IIP Quarterly.xlsx
HK IMF IIP Quarterly.xlsx
IN IMF IIP Quarterly.xlsx
ID IMF IIP Quarterly.xlsx
KR IMF IIP Quarterly.xlsx
MY IMF IIP Quarterly.xlsx
MG IMF IIP Quarterly.xlsx
NP IMF IIP Quarterly.xlsx
PH IMF IIP Quarterly.xlsx
SG IMF IIP Quarterly.xlsx
LK IMF IIP Quarterly.xlsx
TH IMF IIP Quarterly.xlsx


,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00
0,International Investment Assets,Brunei,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,International Investment Assets,Cambodia,USD mn,2025-09-16,2.417597e+04,2.507544e+04,2.677201e+04,2.862880e+04,2.878970e+04,3.065766e+04,...,3.418220e+04,3.729671e+04,3.760191e+04,3.899669e+04,NaN,NaN,NaN,NaN,NaN,NaN
2,International Investment Assets,China,USD mn,2025-10-10,7.503297e+06,7.592054e+06,7.613220e+06,7.846418e+06,7.777337e+06,8.014177e+06,...,9.853377e+06,1.028403e+07,1.021671e+07,1.069780e+07,1.106450e+07,NaN,NaN,NaN,NaN,NaN
3,International Investment Assets,Hong Kong SAR (China),USD mn,2025-10-15,5.678545e+06,5.533903e+06,5.438994e+06,5.641767e+06,5.543582e+06,5.706870e+06,...,6.470570e+06,6.818797e+06,6.763809e+06,7.003228e+06,7.290879e+06,NaN,NaN,NaN,NaN,NaN
4,International Investment Assets,India,USD mn,2025-10-14,6.421384e+05,6.625843e+05,6.702019e+05,6.980954e+05,7.168001e+05,7.487695e+05,...,1.048764e+06,1.116557e+06,1.075463e+06,1.136654e+06,1.186973e+06,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,Portfolio Equity Liabilities,Singapore,USD mn,2025-10-15,2.132782e+05,2.180434e+05,2.153151e+05,2.351231e+05,2.056416e+05,2.180810e+05,...,3.606389e+05,3.986313e+05,3.821894e+05,3.946658e+05,4.036561e+05,NaN,NaN,NaN,NaN,NaN
604,Portfolio Equity Liabilities,Sri Lanka,USD mn,2025-09-29,7.385835e+02,6.513650e+02,9.751852e+02,1.055937e+03,1.990319e+02,NaN,...,1.390010e+03,1.350918e+03,7.623197e+02,7.703150e+02,NaN,NaN,NaN,NaN,NaN,NaN
605,Portfolio Equity Liabilities,Taiwan,USD mn,2025-06-16,4.953840e+05,4.953840e+05,4.953840e+05,4.953840e+05,6.768810e+05,6.768810e+05,...,9.821360e+05,9.821360e+05,9.821360e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
606,Portfolio Equity Liabilities,Thailand,USD mn,2025-10-10,1.182496e+05,1.275366e+05,1.200384e+05,1.161480e+05,7.432922e+04,8.907222e+04,...,7.595472e+04,8.591900e+04,8.139353e+04,7.016220e+04,7.064070e+04,NaN,NaN,NaN,NaN,NaN


In [ ]:
# this is for bop

df_temp2 = pd.DataFrame(columns=y1.columns)
temp1 = pd.DataFrame(columns=y1.columns)
index_list = []
date_use = {'Cambodia': [datetime(2025, 6, 1)],
            'Hong Kong SAR (China)': [datetime(2025, 6, 1)],
            'Malaysia': [datetime(2025, 3, 1), datetime(2025, 6, 1)],
            'Mongolia': [datetime(2025, 6, 1)],
            'Nepal': [datetime(2025, 6, 1)],
            'Sri Lanka': [datetime(2025, 6, 1)],
            'Vietnam': [datetime(2025, 3, 1), datetime(2025, 6, 1)],}

for x,y in zip(df_all_annual_2['Type'].values, df_all_annual_2['Region'].values):
    if y not in date_use.keys():
        continue
    temp1 = y1[(y1['Type'] == x) & (y1['Region'] == y)]
    index_list.append(temp1.index[0])
    temp1.sort_values(by=['Type', 'Region'], inplace=True)
    temp1[date_use[y]] = df_all_annual_2[(df_all_annual_2['Type'] == x) & (df_all_annual_2['Region'] == y)][date_use[y]].values
    df_temp2 = pd.concat([df_temp2, temp1], axis=0)

df_temp2.sort_values(by=['Type', 'Region'], inplace=True)
df_temp2.reset_index(drop=True, inplace=True)
df_temp2

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00
0,Currency and Deposits Assets,Hong Kong SAR (China),USD mn,2025-09-24,-103234.304647,5474.234694,-3092.979693,59729.386712,25683.036672,-42708.044890,...,10338.136858,4845.207060,-15193.038110,25525.831049,1724.540003,NaN,NaN,NaN,NaN,NaN
1,Currency and Deposits Assets,Mongolia,USD mn,2025-09-24,NaN,NaN,NaN,NaN,NaN,NaN,...,437.563986,271.060632,-15.326249,-38.292040,494.705619,NaN,NaN,NaN,NaN,NaN
2,Currency and Deposits Assets,Nepal,USD mn,2025-09-24,132.828355,-154.652067,65.782198,39.918438,97.271918,131.699720,...,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
3,Currency and Deposits Assets,Sri Lanka,USD mn,2025-09-29,150.490533,-109.688302,-74.096829,-59.384304,86.138213,-64.630917,...,-67.642928,-186.315208,208.014514,450.801187,-717.480433,NaN,NaN,NaN,NaN,NaN
4,Currency and Deposits Liabilities,Hong Kong SAR (China),USD mn,2025-09-24,-63358.848016,-4612.627551,-2373.919707,30929.258944,11327.171349,-26468.289117,...,-7843.359625,14831.531262,-14517.726240,6737.662783,22087.412229,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,Trade Credits and Advances Liabilities,Hong Kong SAR (China),USD mn,2025-09-24,-3294.622377,734.438776,361.062625,763.032368,-1957.237830,1330.438148,...,1120.827116,1646.224198,1892.313628,-335.032557,2164.291938,NaN,NaN,NaN,NaN,NaN
152,Trade Credits and Advances Liabilities,Mongolia,USD mn,2025-09-24,NaN,NaN,NaN,NaN,NaN,NaN,...,85.602089,-132.790446,85.693632,-158.818778,472.929916,NaN,NaN,NaN,NaN,NaN
153,Trade Credits and Advances Liabilities,Nepal,USD mn,2025-09-24,4.539566,137.134460,-150.439504,40.367816,418.808560,384.103082,...,665.327220,-0.126418,124.711165,258.171537,50.842653,NaN,NaN,NaN,NaN,NaN
154,Trade Credits and Advances Liabilities,Sri Lanka,USD mn,2025-09-29,-21.851946,105.148054,-25.851946,-85.851946,131.717658,-379.282342,...,-13.800000,-38.000000,-82.000000,-16.200000,-35.000000,NaN,NaN,NaN,NaN,NaN


In [15]:
# this is for IIP

df_temp2 = pd.DataFrame(columns=y1.columns)
temp1 = pd.DataFrame(columns=y1.columns)
index_list = []
date_use = {'Cambodia': [datetime(2025, 6, 1)],
            'Malaysia': [datetime(2025, 3, 1), datetime(2025, 6, 1)],
            'Sri Lanka': [datetime(2025, 6, 1)],}

for x,y in zip(df_all_annual_2['Type'].values, df_all_annual_2['Region'].values):
    if y not in date_use.keys():
        continue
    temp1 = y1[(y1['Type'] == x) & (y1['Region'] == y)]
    index_list.append(temp1.index[0])
    temp1.sort_values(by=['Type', 'Region'], inplace=True)
    temp1[date_use[y]] = df_all_annual_2[(df_all_annual_2['Type'] == x) & (df_all_annual_2['Region'] == y)][date_use[y]].values
    df_temp2 = pd.concat([df_temp2, temp1], axis=0)

df_temp2.sort_values(by=['Type', 'Region'], inplace=True)
df_temp2.reset_index(drop=True, inplace=True)
df_temp2

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00
0,Currency and Deposits Assets,Cambodia,USD mn,2025-09-16,6142.117808,5906.682665,6344.549294,6801.412321,7416.579029,7934.776442,...,10490.197337,12040.316023,10637.103001,11529.243249,11824.435969,NaN,NaN,NaN,NaN,NaN
1,Currency and Deposits Liabilities,Cambodia,USD mn,2025-09-16,2546.981616,2670.133914,3024.243508,3412.538928,3509.043966,3547.989230,...,4078.596142,3957.038607,3937.365143,3949.635128,3856.245683,NaN,NaN,NaN,NaN,NaN
2,FDI Assets,Cambodia,USD mn,2025-09-16,871.803862,901.219618,931.105308,951.230310,979.508508,983.442479,...,1542.416323,1605.003927,1644.386643,1684.247109,1720.904391,NaN,NaN,NaN,NaN,NaN
3,FDI Assets,Malaysia,USD mn,2025-09-16,146858.160416,148069.902002,142315.781809,142694.222127,140526.475090,142980.721808,...,173155.096373,182896.849193,178047.235340,180304.167610,188303.936394,NaN,NaN,NaN,NaN,NaN
4,FDI Assets,Sri Lanka,USD mn,2025-09-29,1448.837048,1465.005244,1481.173440,1497.341636,1504.042332,NaN,...,1648.136796,1679.869534,1698.995756,1716.179580,1739.538010,NaN,NaN,NaN,NaN,NaN
5,FDI Debt Assets,Malaysia,USD mn,2025-09-16,69323.097394,67703.520143,62649.923954,63722.813414,60764.092887,61249.562163,...,75801.754445,78833.112946,78396.119500,80139.019544,81799.320669,NaN,NaN,NaN,NaN,NaN
6,FDI Debt Liabilities,Cambodia,USD mn,2025-09-16,426.225916,426.225916,426.225916,426.225916,426.225916,426.225916,...,426.225916,426.225916,426.225916,426.225916,427.085938,NaN,NaN,NaN,NaN,NaN
7,FDI Debt Liabilities,Malaysia,USD mn,2025-09-16,42244.062624,43260.786868,38568.249422,39548.504993,39396.209713,39834.216041,...,62386.440795,71127.168482,74386.890257,78287.879986,82846.325405,NaN,NaN,NaN,NaN,NaN
8,FDI Equity Assets,Cambodia,USD mn,2025-09-16,871.803862,901.219618,931.105308,951.230310,979.508508,983.442479,...,1542.416323,1605.003927,1644.386643,1684.247109,1720.904391,NaN,NaN,NaN,NaN,NaN
9,FDI Equity Assets,Malaysia,USD mn,2025-09-16,77535.063022,80366.381860,79665.857855,78971.408713,79762.382203,81731.159645,...,97353.341928,104063.736247,99651.115840,100165.148066,106504.615726,NaN,NaN,NaN,NaN,NaN


In [16]:
y1.drop(index_list, inplace=True)
y1 = pd.concat([y1, df_temp2], axis=0)
y1.sort_values(by=['Type', 'Region'], inplace=True)
y1.reset_index(drop=True, inplace=True)
y1

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00
0,Currency and Deposits Assets,Brunei,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Currency and Deposits Assets,Cambodia,USD mn,2025-09-16,6142.117808,5906.682665,6344.549294,6801.412321,7416.579029,7934.776442,...,1.049020e+04,1.204032e+04,1.063710e+04,1.152924e+04,1.182444e+04,NaN,NaN,NaN,NaN,NaN
2,Currency and Deposits Assets,China,USD mn,2025-10-10,373380.182787,390054.216860,385949.008138,396160.768530,389879.774326,410682.044666,...,5.171333e+05,5.416339e+05,5.380273e+05,5.603557e+05,5.929571e+05,NaN,NaN,NaN,NaN,NaN
3,Currency and Deposits Assets,Hong Kong SAR (China),USD mn,2025-10-15,838782.265257,847503.712238,842203.774547,908993.707461,936036.879433,898428.433206,...,1.147299e+06,1.170692e+06,1.151990e+06,1.182342e+06,1.187634e+06,NaN,NaN,NaN,NaN,NaN
4,Currency and Deposits Assets,India,USD mn,2025-10-14,25157.515226,24176.848631,27516.533906,27034.380201,26011.129710,27736.129926,...,5.774715e+04,5.610496e+04,6.863036e+04,7.933201e+04,8.252844e+04,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,Trade Credits and Advances Liabilities,Singapore,USD mn,2025-10-15,88909.506601,108614.554858,107049.590965,98820.887767,107380.501158,99372.093023,...,1.290130e+05,1.359497e+05,1.306531e+05,1.280206e+05,1.359216e+05,NaN,NaN,NaN,NaN,NaN
604,Trade Credits and Advances Liabilities,Sri Lanka,USD mn,2025-09-29,2162.545718,2267.693773,2241.841827,2155.989882,2070.489882,NaN,...,7.095747e+02,6.715747e+02,5.895747e+02,5.733747e+02,NaN,NaN,NaN,NaN,NaN,NaN
605,Trade Credits and Advances Liabilities,Taiwan,USD mn,2025-06-16,91198.000000,91198.000000,91198.000000,91198.000000,100204.000000,100204.000000,...,9.074200e+04,9.074200e+04,9.074200e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN
606,Trade Credits and Advances Liabilities,Thailand,USD mn,2025-10-10,20478.755731,20385.578822,19002.880378,18958.938449,19388.393806,19153.069679,...,2.837543e+04,3.035445e+04,2.981241e+04,2.935369e+04,3.081975e+04,NaN,NaN,NaN,NaN,NaN


In [17]:
y1.to_excel(r"section 2 IIP data_updated.xlsx", sheet_name='IIP Quarterly', index=False)

,Date,Portfolio Debt,Portfolio Equity
0,2000-01-15,-0.3207,3.27702
1,2000-02-15,0.6629,3.49665
2,2000-03-15,1.4968,5.73722
3,2000-04-15,0.6739,0.01669
4,2000-05-15,-1.6931,1.10859
...,...,...,...
305,2025-06-15,19.542106,13.594164
306,2025-07-15,-10.364947,22.147776
307,2025-08-15,12.155782,8.134126
308,2025-09-15,15.302716,2.467315


In [ ]:
with pd.ExcelWriter('section 2 v2.xlsx') as writer:
    quarterly_type_half.to_excel(writer, sheet_name='quarterly_type_half', index=False)
    quarterly_type_quarter.to_excel(writer, sheet_name='quarterly_type_quarter', index=False)

,China,India,Indonesia,Korea,Malaysia,Philippines,"Taiwan, China",Thailand,Vietnam,Sri lanka,Mongolia
Date,,,,,,,,,,,
2000-01-15,NaN,NaN,NaN,1416.7,NaN,NaN,1860.32,NaN,NaN,NaN,NaN
2000-02-15,NaN,NaN,NaN,1716.8,NaN,NaN,1779.85,NaN,NaN,NaN,NaN
2000-03-15,NaN,NaN,NaN,3665.3,NaN,NaN,2071.92,NaN,NaN,NaN,NaN
2000-04-15,NaN,NaN,NaN,235.5,NaN,NaN,-218.81,NaN,NaN,NaN,NaN
2000-05-15,NaN,NaN,NaN,786.5,NaN,NaN,322.09,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2025-06-15,5444.425179,2372.67,-511.497,2009.18,-305.3,-83.9341,4962.88,-243.86,-44.31,-6.8,0.71
2025-07-15,12151.7039,-2852.21,-511.264,4517.35,-221.3,-28.8641,8274.09,499.3,297.58,-19.7,41.09
2025-08-15,18193.672206,-4313.59,675.513,-1059.97,-812.9,-73.7895,-2246.01,-670.22,-1543.37,-15.2,-0.01


In [4]:
from data_section_2 import iip_quarterly, cb_iip_quarterly
import pandas as pd

x1, y1, z1 = iip_quarterly()
y2 = cb_iip_quarterly()

y1

KH IMF IIP Quarterly.xlsx
CH IMF IIP Quarterly.xlsx
HK IMF IIP Quarterly.xlsx
IN IMF IIP Quarterly.xlsx
ID IMF IIP Quarterly.xlsx
KR IMF IIP Quarterly.xlsx
MY IMF IIP Quarterly.xlsx
MG IMF IIP Quarterly.xlsx
NP IMF IIP Quarterly.xlsx
PH IMF IIP Quarterly.xlsx
SG IMF IIP Quarterly.xlsx
LK IMF IIP Quarterly.xlsx
TH IMF IIP Quarterly.xlsx
KH CB IIP Quarterly.xlsx
CH CB IIP Quarterly.xlsx
HK CB IIP Quarterly.xlsx
IN CB IIP Quarterly.xlsx
ID CB IIP Quarterly.xlsx
KR CB IIP Quarterly.xlsx
MY CB IIP Quarterly.xlsx
MG CB IIP Quarterly.xlsx
PH CB IIP Quarterly.xlsx
SG CB IIP Quarterly.xlsx
LK CB IIP Quarterly.xlsx
TH CB IIP Quarterly.xlsx


,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00
0,International Investment Assets,Brunei,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,International Investment Assets,Cambodia,USD mn,2025-09-16,2.417597e+04,2.507544e+04,2.677201e+04,2.862880e+04,2.878970e+04,3.065766e+04,...,3.418220e+04,3.729671e+04,3.760191e+04,3.899669e+04,NaN,NaN,NaN,NaN,NaN,NaN
2,International Investment Assets,China,USD mn,2025-10-10,7.503297e+06,7.592054e+06,7.613220e+06,7.846418e+06,7.777337e+06,8.014177e+06,...,9.853377e+06,1.028403e+07,1.021671e+07,1.069780e+07,1.106450e+07,NaN,NaN,NaN,NaN,NaN
3,International Investment Assets,Hong Kong SAR (China),USD mn,2025-10-15,5.678545e+06,5.533903e+06,5.438994e+06,5.641767e+06,5.543582e+06,5.706870e+06,...,6.470570e+06,6.818797e+06,6.763809e+06,7.003228e+06,7.290879e+06,NaN,NaN,NaN,NaN,NaN
4,International Investment Assets,India,USD mn,2025-10-14,6.421384e+05,6.625843e+05,6.702019e+05,6.980954e+05,7.168001e+05,7.487695e+05,...,1.048764e+06,1.116557e+06,1.075463e+06,1.136654e+06,1.186973e+06,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,Portfolio Equity Liabilities,Singapore,USD mn,2025-10-15,2.132782e+05,2.180434e+05,2.153151e+05,2.351231e+05,2.056416e+05,2.180810e+05,...,3.606389e+05,3.986313e+05,3.821894e+05,3.946658e+05,4.036561e+05,NaN,NaN,NaN,NaN,NaN
604,Portfolio Equity Liabilities,Sri Lanka,USD mn,2025-09-29,7.385835e+02,6.513650e+02,9.751852e+02,1.055937e+03,1.990319e+02,NaN,...,1.390010e+03,1.350918e+03,7.623197e+02,7.703150e+02,NaN,NaN,NaN,NaN,NaN,NaN
605,Portfolio Equity Liabilities,Taiwan,USD mn,2025-06-16,4.953840e+05,4.953840e+05,4.953840e+05,4.953840e+05,6.768810e+05,6.768810e+05,...,9.821360e+05,9.821360e+05,9.821360e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
606,Portfolio Equity Liabilities,Thailand,USD mn,2025-10-10,1.182496e+05,1.275366e+05,1.200384e+05,1.161480e+05,7.432922e+04,8.907222e+04,...,7.595472e+04,8.591900e+04,8.139353e+04,7.016220e+04,7.064070e+04,NaN,NaN,NaN,NaN,NaN


In [5]:
y2.insert(loc=0, column='Group', value='CB')
y1.insert(loc=0, column='Group', value='IMF')

dict1 = dict()
for x in y2['Region'].unique():
    dict1[x] = y2[y2.Region == x].Type.tolist()

df_temp = pd.DataFrame(columns=y1.columns)
for x,y in dict1.items():
    temp1 = y1[y1.Region == x]
    temp1 = temp1[temp1.Type.isin(y)]
    df_temp = pd.concat([df_temp, temp1], axis=0)

y2 = pd.concat([y2, df_temp], axis=0)
sort_col = y2.columns[:5].tolist() + y1.columns[5:].sort_values(ascending=True).to_list()
y2 = y2[sort_col]
y2.sort_values(by=['Type', 'Region'], inplace=True)

In [ ]:
y2.to_excel(r"section 2 IIP compare.xlsx", sheet_name='BOP Quarterly', index=False)

In [14]:
import pandas as pd
y2 = pd.concat([y2, y1], axis=0)
y2.sort_values(by=['Type', 'Region'], inplace=True)
y2

,Group,Type,Region,Unit,Last Update Time,2021-12-01 00:00:00,2022-03-01 00:00:00,2022-06-01 00:00:00,2022-09-01 00:00:00,2022-12-01 00:00:00,...,2020-09-01 00:00:00,2020-12-01 00:00:00,2021-03-01 00:00:00,2021-06-01 00:00:00,2021-09-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00
286,IMF,Currency and Deposits Assets,Cambodia,USD mn,2025-09-24,-1300.077557,-579.590324,-55.964981,567.736609,-393.257617,...,1323.192668,947.467624,-1962.776759,-1361.852205,-448.561534,NaN,NaN,NaN,NaN,NaN
0,CB,Currency and Deposits Assets,Hong Kong SAR (China),USD mn,2025-09-19,12961.755228,19474.239441,6679.057714,2587.057070,-9920.992074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
288,IMF,Currency and Deposits Assets,Hong Kong SAR (China),USD mn,2025-09-24,12960.590501,19470.663592,6678.064735,2586.826347,-9919.301236,...,49650.412797,19389.913582,3591.776231,23806.850373,10532.973390,NaN,NaN,NaN,NaN,NaN
293,IMF,Currency and Deposits Assets,Malaysia,USD mn,2025-09-24,5116.347527,959.420539,-1158.127370,6313.506645,-4068.639953,...,-6163.789647,-695.116224,5999.146693,-3975.473178,1667.899902,NaN,NaN,NaN,NaN,NaN
1,CB,Currency and Deposits Assets,Mongolia,USD mn,2025-10-31,-26.614700,139.194332,-39.012104,144.816935,21.888081,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
543,IMF,Trade Credits and Advances Liabilities,Nepal,USD mn,2025-09-24,108.312350,743.747434,964.966939,-61.260996,164.123344,...,231.546305,333.888426,359.372005,65.630798,416.780461,NaN,NaN,NaN,NaN,NaN
154,CB,Trade Credits and Advances Liabilities,Sri Lanka,USD mn,2025-10-01,-746.254118,-296.617618,-138.733208,-195.918812,-263.785681,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
547,IMF,Trade Credits and Advances Liabilities,Sri Lanka,USD mn,2025-09-29,-746.254118,-296.617618,-138.733208,-195.918812,-263.785681,...,131.717658,300.717658,213.000000,-80.000000,187.000000,NaN,NaN,NaN,NaN,NaN
155,CB,Trade Credits and Advances Liabilities,Vietnam,USD mn,2024-09-30,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
from data_section_2 import to_combine_iipq, to_combine_bopq

df = to_combine_iipq(year_use=2023)
df2 = to_combine_bopq(year_use=2023)
df

KH IMF IIP Quarterly.xlsx
CH IMF IIP Quarterly.xlsx
HK IMF IIP Quarterly.xlsx
IN IMF IIP Quarterly.xlsx
ID IMF IIP Quarterly.xlsx
KR IMF IIP Quarterly.xlsx
MY IMF IIP Quarterly.xlsx
MG IMF IIP Quarterly.xlsx
NP IMF IIP Quarterly.xlsx
PH IMF IIP Quarterly.xlsx
SG IMF IIP Quarterly.xlsx
LK IMF IIP Quarterly.xlsx
TH IMF IIP Quarterly.xlsx
KH CB IIP Quarterly.xlsx
CH CB IIP Quarterly.xlsx
HK CB IIP Quarterly.xlsx
IN CB IIP Quarterly.xlsx
ID CB IIP Quarterly.xlsx
KR CB IIP Quarterly.xlsx
MY CB IIP Quarterly.xlsx
MG CB IIP Quarterly.xlsx
PH CB IIP Quarterly.xlsx
SG CB IIP Quarterly.xlsx
LK CB IIP Quarterly.xlsx
TH CB IIP Quarterly.xlsx
BN IMF BOP Quarterly.xlsx
KH IMF BOP Quarterly.xlsx
CH IMF BOP Quarterly.xlsx
HK IMF BOP Quarterly.xlsx
IN IMF BOP Quarterly.xlsx
ID IMF BOP Quarterly.xlsx
KR IMF BOP Quarterly.xlsx
LA IMF BOP Quarterly.xlsx
MY IMF BOP Quarterly.xlsx
MG IMF BOP Quarterly.xlsx
NP IMF BOP Quarterly.xlsx
PG IMF BOP Quarterly.xlsx
PH IMF BOP Quarterly.xlsx
SG IMF BOP Quarterly.xls

ValueError: Length of values (2) does not match length of index (1)

In [ ]:
df2

In [1]:
from data_section_2 import to_combine_bopq, to_combine_iipq

df1 = to_combine_bopq(year_use=2023)
df2 = to_combine_iipq(year_use=2023)
df1

BN IMF BOP Quarterly.xlsx
KH IMF BOP Quarterly.xlsx
CH IMF BOP Quarterly.xlsx
HK IMF BOP Quarterly.xlsx
IN IMF BOP Quarterly.xlsx
ID IMF BOP Quarterly.xlsx
KR IMF BOP Quarterly.xlsx
LA IMF BOP Quarterly.xlsx
MY IMF BOP Quarterly.xlsx
MG IMF BOP Quarterly.xlsx
NP IMF BOP Quarterly.xlsx
PG IMF BOP Quarterly.xlsx
PH IMF BOP Quarterly.xlsx
SG IMF BOP Quarterly.xlsx
LK IMF BOP Quarterly.xlsx
TH IMF BOP Quarterly.xlsx
VN IMF BOP Quarterly.xlsx
KH CB BOP Quarterly.xlsx
CH CB BOP Quarterly.xlsx
TW CB BOP Quarterly.xlsx
HK CB BOP Quarterly.xlsx
IN CB BOP Quarterly.xlsx
ID CB BOP Quarterly.xlsx
LA CB BOP Quarterly.xlsx
MY CB BOP Quarterly.xlsx
MG CB BOP Quarterly.xlsx
MM CB BOP Quarterly.xlsx
NP CB BOP Quarterly.xlsx
SG CB BOP Quarterly.xlsx
LK CB BOP Quarterly.xlsx
TH CB BOP Quarterly.xlsx
VN CB BOP Quarterly.xlsx
KH IMF IIP Quarterly.xlsx
CH IMF IIP Quarterly.xlsx
HK IMF IIP Quarterly.xlsx
IN IMF IIP Quarterly.xlsx
ID IMF IIP Quarterly.xlsx
KR IMF IIP Quarterly.xlsx
MY IMF IIP Quarterly.xlsx
M

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00,2026-12-01 00:00:00
0,Currency and Deposits Assets,Brunei,USD mn,2025-09-24,NaN,NaN,NaN,NaN,547.766981,-967.856282,...,-271.822505,233.128069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Currency and Deposits Assets,Cambodia,USD mn,2026-01-08,-155.233357,-106.511662,207.458354,446.565258,623.615746,588.665969,...,1466.995765,-900.269701,1503.904419,76.381670,1426.648820,NaN,NaN,NaN,NaN,NaN
2,Currency and Deposits Assets,China,USD mn,2026-01-16,24955.924265,30010.361268,13896.411204,32887.004398,3083.992791,32548.670479,...,20680.539338,6355.973137,21363.473119,33869.042669,-27422.039443,NaN,NaN,NaN,NaN,NaN
3,Currency and Deposits Assets,Hong Kong SAR (China),USD mn,2026-01-21,-103234.304647,5474.234694,-3092.979693,59729.386712,25683.036672,-42708.044890,...,14888.841403,-11584.472928,25525.831049,1724.540003,46588.092397,NaN,NaN,NaN,NaN,NaN
4,Currency and Deposits Assets,India,USD mn,2025-12-19,11564.300849,8409.083534,11513.296823,13637.167663,14079.099404,8419.967249,...,22753.392913,22776.052110,23676.898542,20163.632000,20928.183437,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660,Trade Credits and Advances Liabilities,Singapore,USD mn,2025-12-04,230.338082,3245.604920,2127.782975,-731.862256,17089.103440,-7512.272341,...,-301.724591,3201.356174,-1060.705705,543.818895,501.335428,NaN,NaN,NaN,NaN,NaN
661,Trade Credits and Advances Liabilities,Sri Lanka,USD mn,2025-09-29,-21.851946,105.148054,-25.851946,-85.851946,131.717658,-379.282342,...,-38.000000,-82.000000,-16.200000,-35.000000,-28.800000,NaN,NaN,NaN,NaN,NaN
662,Trade Credits and Advances Liabilities,Taiwan,USD mn,2025-11-20,1175.000000,1176.000000,-418.000000,1658.000000,3519.000000,2316.000000,...,-2797.000000,644.000000,835.000000,-1885.000000,-5705.000000,NaN,NaN,NaN,NaN,NaN
663,Trade Credits and Advances Liabilities,Thailand,USD mn,2026-01-26,-347.525135,-181.348372,-1362.283164,-80.043961,552.942734,-72.154305,...,1660.929607,-313.951858,-1204.778272,183.562313,2826.561469,NaN,NaN,NaN,NaN,NaN


In [3]:
import pandas as pd

with pd.ExcelWriter('section 2_all_data (IMF & CB).xlsx') as writer:
    df1.to_excel(writer, sheet_name='BOP Quarterly', index=False)
    df2.to_excel(writer, sheet_name='IIP Quarterly', index=False)

In [69]:
# # combine imf and cb
# df_temp2[(df_temp2['Region'] == 'Hong Kong SAR (China)') & (df_temp2['Type'] == 'FDI Assets')]

y1.drop(index_list, inplace=True, axis=0)
y1

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00,2026-12-01 00:00:00
0,Current Account,Brunei,USD mn,2025-09-24,NaN,NaN,NaN,NaN,1155.536549,363.959828,...,253.547381,543.632218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Current Account,Cambodia,USD mn,2026-01-08,-558.380622,-1106.225081,-479.117277,-791.075903,-677.665626,68.428174,...,1027.414183,-811.507667,29.692083,-75.512138,79.203213,NaN,NaN,NaN,NaN,NaN
2,Current Account,China,USD mn,2026-01-16,22297.890144,19812.644962,30634.105981,30165.234921,-52251.501318,91091.595723,...,157436.502296,163775.570723,165448.875384,128677.100032,198677.500152,NaN,NaN,NaN,NaN,NaN
4,Current Account,India,USD mn,2025-12-19,-4628.370364,-14976.547068,-7553.331364,-2604.615854,584.382738,19083.025662,...,-20832.318926,-11314.930136,13654.435758,-2722.203923,-12292.430828,NaN,NaN,NaN,NaN,NaN
5,Current Account,Indonesia,USD mn,2025-11-21,-6550.291346,-8199.411300,-7480.918280,-8048.438827,-3366.708107,-2913.208589,...,-2017.481630,-1292.966681,-171.845751,-2748.417149,4047.454620,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
659,Portfolio Equity Liabilities,Philippines,USD mn,2025-12-15,1896.916040,522.360537,-313.130412,-342.227991,-591.806964,-729.791075,...,632.550364,-473.945719,-294.591188,-144.131487,-17.916492,NaN,NaN,NaN,NaN,NaN
660,Portfolio Equity Liabilities,Singapore,USD mn,2025-12-04,927.079796,3151.037948,-1070.428038,1135.569899,-658.701708,-1590.739980,...,7060.485452,5130.837129,4307.717767,-2925.668383,2613.020877,NaN,NaN,NaN,NaN,NaN
662,Portfolio Equity Liabilities,Taiwan,USD mn,2025-11-20,3133.000000,-650.000000,-3776.000000,9403.000000,-16326.000000,-1759.000000,...,-16714.000000,-7119.000000,-20261.000000,14136.000000,5740.000000,NaN,NaN,NaN,NaN,NaN
663,Portfolio Equity Liabilities,Thailand,USD mn,2026-01-26,-400.887822,1240.793707,-1116.975851,332.900270,-3651.185970,-3005.685278,...,1315.281433,-3478.960505,-306.260984,655.999050,117.911222,NaN,NaN,NaN,NaN,NaN


In [ ]:
# IMF
y1[(y1['Region'] == 'Hong Kong SAR (China)') & (y1['Type'] == 'FDI Assets')]

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00,2026-12-01 00:00:00
98,FDI Assets,Hong Kong SAR (China),USD mn,2026-01-21,34290.37465,-31017.984694,17226.659287,17268.90971,34497.362213,12740.938212,...,18153.083465,6459.639045,43185.143934,24981.811195,19772.076372,NaN,NaN,NaN,NaN,NaN


In [ ]:
# CB
y2[(y2['Region'] == 'Hong Kong SAR (China)') & (y2['Type'] == 'FDI Assets')] 

,Type,Region,Unit,Last Update Time,2021-12-01 00:00:00,2022-03-01 00:00:00,2022-06-01 00:00:00,2022-09-01 00:00:00,2022-12-01 00:00:00,2023-03-01 00:00:00,2023-06-01 00:00:00,2023-09-01 00:00:00,2023-12-01 00:00:00,2024-03-01 00:00:00,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00
25,FDI Assets,Hong Kong SAR (China),USD mn,2025-09-19,16832.578024,50816.146102,31781.124541,15507.816979,20916.261826,28342.716936,20491.668757,6182.553344,41177.100442,31900.542172,9622.042947,13172.626909,23416.795647,43192.730046,24990.133771,NaN


In [8]:
import pandas as pd

with pd.ExcelWriter('section 2_all_data.xlsx') as writer:
    y.to_excel(writer, sheet_name='BOP Quarterly', index=False)
    y1.to_excel(writer, sheet_name='IIP Quarterly', index=False)

In [1]:
from data_section_2 import *

x1, x2, x3 = bop_quarterly()
y1, y2, y3 = iip_quarterly()

BN IMF BOP Quarterly.xlsx
KH IMF BOP Quarterly.xlsx
CH IMF BOP Quarterly.xlsx
HK IMF BOP Quarterly.xlsx
IN IMF BOP Quarterly.xlsx
ID IMF BOP Quarterly.xlsx
KR IMF BOP Quarterly.xlsx
LA IMF BOP Quarterly.xlsx
MY IMF BOP Quarterly.xlsx
MG IMF BOP Quarterly.xlsx
NP IMF BOP Quarterly.xlsx
PG IMF BOP Quarterly.xlsx
PH IMF BOP Quarterly.xlsx
SG IMF BOP Quarterly.xlsx
LK IMF BOP Quarterly.xlsx
TH IMF BOP Quarterly.xlsx
VN IMF BOP Quarterly.xlsx
KH IMF IIP Quarterly.xlsx
CH IMF IIP Quarterly.xlsx
HK IMF IIP Quarterly.xlsx
IN IMF IIP Quarterly.xlsx
ID IMF IIP Quarterly.xlsx
KR IMF IIP Quarterly.xlsx
MY IMF IIP Quarterly.xlsx
MG IMF IIP Quarterly.xlsx
NP IMF IIP Quarterly.xlsx
PH IMF IIP Quarterly.xlsx
SG IMF IIP Quarterly.xlsx
LK IMF IIP Quarterly.xlsx
TH IMF IIP Quarterly.xlsx


In [2]:
import pandas as pd

with pd.ExcelWriter('section 2 IMF data.xlsx') as writer:
    x2.to_excel(writer, sheet_name='BOP Quarterly', index=False)
    y2.to_excel(writer, sheet_name='IIP Quarterly', index=False)

In [3]:
df = to_combine_bopq()
df

BN IMF BOP Quarterly.xlsx
KH IMF BOP Quarterly.xlsx
CH IMF BOP Quarterly.xlsx
HK IMF BOP Quarterly.xlsx
IN IMF BOP Quarterly.xlsx
ID IMF BOP Quarterly.xlsx
KR IMF BOP Quarterly.xlsx
LA IMF BOP Quarterly.xlsx
MY IMF BOP Quarterly.xlsx
MG IMF BOP Quarterly.xlsx
NP IMF BOP Quarterly.xlsx
PG IMF BOP Quarterly.xlsx
PH IMF BOP Quarterly.xlsx
SG IMF BOP Quarterly.xlsx
LK IMF BOP Quarterly.xlsx
TH IMF BOP Quarterly.xlsx
VN IMF BOP Quarterly.xlsx
KH CB BOP Quarterly.xlsx
CH CB BOP Quarterly.xlsx
TW CB BOP Quarterly.xlsx
HK CB BOP Quarterly.xlsx
IN CB BOP Quarterly.xlsx
ID CB BOP Quarterly.xlsx
LA CB BOP Quarterly.xlsx
MY CB BOP Quarterly.xlsx
MG CB BOP Quarterly.xlsx
MM CB BOP Quarterly.xlsx
NP CB BOP Quarterly.xlsx
SG CB BOP Quarterly.xlsx
LK CB BOP Quarterly.xlsx
TH CB BOP Quarterly.xlsx
VN CB BOP Quarterly.xlsx


,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00,2026-12-01 00:00:00
0,Currency and Deposits Assets,Brunei,USD mn,2025-09-24,NaN,NaN,NaN,NaN,547.766981,-967.856282,...,-271.822505,233.128069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Currency and Deposits Assets,Cambodia,USD mn,2026-01-08,-155.233357,-106.511662,207.458354,446.565258,623.615746,588.665969,...,1466.995765,-900.269701,1503.904419,76.381670,1426.648820,NaN,NaN,NaN,NaN,NaN
2,Currency and Deposits Assets,China,USD mn,2026-01-16,24955.924265,30010.361268,13896.411204,32887.004398,3083.992791,32548.670479,...,20680.539338,6355.973137,21363.473119,33869.042669,-27422.039443,NaN,NaN,NaN,NaN,NaN
3,Currency and Deposits Assets,Hong Kong SAR (China),USD mn,2026-01-21,-103234.304647,5474.234694,-3092.979693,59729.386712,25683.036672,-42708.044890,...,14888.841403,-11584.472928,25525.831049,1724.540003,46588.092397,NaN,NaN,NaN,NaN,NaN
4,Currency and Deposits Assets,India,USD mn,2025-12-19,11564.300849,8409.083534,11513.296823,13637.167663,14079.099404,8419.967249,...,22753.392913,22776.052110,23676.898542,20163.632000,20928.183437,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660,Trade Credits and Advances Liabilities,Singapore,USD mn,2026-02-26,239.187274,3247.341190,2124.432353,-661.051782,17089.330767,-7514.674975,...,83.785108,3477.734872,-1070.549733,1666.331770,219.851735,5727.402504,NaN,NaN,NaN,NaN
661,Trade Credits and Advances Liabilities,Sri Lanka,USD mn,2025-09-29,-21.851946,105.148054,-25.851946,-85.851946,131.717658,-379.282342,...,-38.000000,-82.000000,-16.200000,-35.000000,-28.800000,NaN,NaN,NaN,NaN,NaN
662,Trade Credits and Advances Liabilities,Taiwan,USD mn,2026-02-26,1175.000000,1176.000000,-418.000000,1658.000000,3519.000000,2316.000000,...,-2828.000000,624.000000,833.000000,-1878.000000,-5713.000000,-14196.000000,NaN,NaN,NaN,NaN
663,Trade Credits and Advances Liabilities,Thailand,USD mn,2026-01-26,-347.525135,-181.348372,-1362.283164,-80.043961,552.942734,-72.154305,...,1660.929607,-313.951858,-1204.778272,183.562313,2826.561469,NaN,NaN,NaN,NaN,NaN


In [5]:
df.to_excel('bop_quarterly_combined.xlsx', index=False)

In [5]:
import requests
from pprint import pprint

base = "https://stats.bis.org/api/v1"

# Get all datasets
flows = requests.get(f"{base}/dataflow")
pprint(flows.text)

('<?xml version="1.0" ?><mes:Structure '
 'xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" '
 'xmlns:mes="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message" '
 'xmlns:str="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure" '
 'xmlns:com="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common"><mes:Header><mes:ID>IDREFcfb6f716-f9fb-4fde-aa06-0432690d317b</mes:ID><mes:Test>false</mes:Test><mes:Prepared>2026-02-24T03:33:29Z</mes:Prepared><mes:Sender '
 'id="UNKNOWN"></mes:Sender><mes:Receiver '
 'id="not_supplied"></mes:Receiver></mes:Header><mes:Structures><str:Dataflows><str:Dataflow '
 'urn="urn:sdmx:org.sdmx.infomodel.datastructure.Dataflow=BIS:BIS_REL_CAL(1.0)" '
 'isExternalReference="false" agencyID="BIS" id="BIS_REL_CAL" isFinal="false" '
 'version="1.0"><com:Name '
 'xml:lang="en">BIS_RELEASE_CALENDAR</com:Name><str:Structure><Ref '
 'package="datastructure" agencyID="BIS" id="BIS_RELEASE_CALENDAR" '
 'version="1.0" '
 'class="DataStructure"></Ref></str:

In [8]:
y1

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00,2026-06-01 00:00:00,2026-09-01 00:00:00,2026-12-01 00:00:00
0,Current Account,Brunei,USD mn,2025-09-24,NaN,NaN,NaN,NaN,1155.536549,363.959828,...,253.547381,543.632218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Current Account,Cambodia,USD mn,2026-01-08,-558.380622,-1106.225081,-479.117277,-791.075903,-677.665626,68.428174,...,1027.414183,-811.507667,29.692083,-75.512138,79.203213,NaN,NaN,NaN,NaN,NaN
2,Current Account,China,USD mn,2026-01-16,22297.890144,19812.644962,30634.105981,30165.234921,-52251.501318,91091.595723,...,157436.502296,163775.570723,165448.875384,128677.100032,198677.500152,NaN,NaN,NaN,NaN,NaN
3,Current Account,Hong Kong SAR (China),USD mn,2026-01-21,3083.977572,3823.469388,8717.357061,5628.321976,-902.123097,7955.798254,...,14517.030642,14356.711107,16310.829335,12272.063533,12552.207637,NaN,NaN,NaN,NaN,NaN
4,Current Account,India,USD mn,2025-12-19,-4628.370364,-14976.547068,-7553.331364,-2604.615854,584.382738,19083.025662,...,-20832.318926,-11314.930136,13654.435758,-2722.203923,-12292.430828,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660,Portfolio Equity Liabilities,Singapore,USD mn,2025-12-04,927.079796,3151.037948,-1070.428038,1135.569899,-658.701708,-1590.739980,...,7060.485452,5130.837129,4307.717767,-2925.668383,2613.020877,NaN,NaN,NaN,NaN,NaN
661,Portfolio Equity Liabilities,Sri Lanka,USD mn,2025-09-29,-13.525046,23.019609,35.332527,-49.194841,-16.437626,-86.270250,...,26.959144,-9.087287,-80.325508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
662,Portfolio Equity Liabilities,Taiwan,USD mn,2025-11-20,3133.000000,-650.000000,-3776.000000,9403.000000,-16326.000000,-1759.000000,...,-16714.000000,-7119.000000,-20261.000000,14136.000000,5740.000000,NaN,NaN,NaN,NaN,NaN
663,Portfolio Equity Liabilities,Thailand,USD mn,2026-01-26,-400.887822,1240.793707,-1116.975851,332.900270,-3651.185970,-3005.685278,...,1315.281433,-3478.960505,-306.260984,655.999050,117.911222,NaN,NaN,NaN,NaN,NaN


In [19]:
from datetime import datetime

y1[datetime(2019, 3, 1)]

0               NaN
1       -558.380622
2      22297.890144
3       3083.977572
4      -4628.370364
           ...     
660      927.079796
661      -13.525046
662     3133.000000
663     -400.887822
664             NaN
Name: 2019-03-01 00:00:00, Length: 665, dtype: float64

In [27]:
# section 2 CFM data export to excel

import pandas as pd
import xlwings as xw
from datetime import datetime
# from data_section_2 import *

# # parameter
start_date = datetime(2023, 3, 1)
# x,y,z = bop_quarterly()
index = y1.columns.tolist().index(start_date)
df = y1.copy()[y1.columns[:4].tolist()+y1.columns[index:index+11].tolist()]
df

,Type,Region,Unit,Last Update Time,2023-03-01 00:00:00,2023-06-01 00:00:00,2023-09-01 00:00:00,2023-12-01 00:00:00,2024-03-01 00:00:00,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00
0,Current Account,Brunei,USD mn,2025-09-24,799.180981,-21.363025,491.872688,673.950972,811.096665,621.378367,253.547381,543.632218,NaN,NaN,NaN
1,Current Account,Cambodia,USD mn,2026-01-08,-131.770663,418.100938,353.228582,-87.212607,519.841199,-507.646839,1027.414183,-811.507667,29.692083,-75.512138,79.203213
2,Current Account,China,USD mn,2026-01-16,73870.955972,62544.622141,68490.370468,58476.404786,47247.937881,55459.203817,157436.502296,163775.570723,165448.875384,128677.100032,198677.500152
3,Current Account,Hong Kong SAR (China),USD mn,2026-01-21,5337.953908,4963.436929,14171.558812,7865.034339,13221.223098,12375.015988,14517.030642,14356.711107,16310.829335,12272.063533,12552.207637
4,Current Account,India,USD mn,2025-12-19,-1336.195578,-8941.834746,-11262.112327,-10415.355279,4586.133440,-4454.302806,-20832.318926,-11314.930136,13654.435758,-2722.203923,-12292.430828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660,Portfolio Equity Liabilities,Singapore,USD mn,2025-12-04,-5040.034522,-283.840639,-1507.760626,8916.396928,6305.193154,1564.252038,7060.485452,5130.837129,4307.717767,-2925.668383,2613.020877
661,Portfolio Equity Liabilities,Sri Lanka,USD mn,2025-09-29,3.913408,0.876198,4.605242,-1.305607,-13.965964,-2.308724,26.959144,-9.087287,-80.325508,NaN,NaN
662,Portfolio Equity Liabilities,Taiwan,USD mn,2025-11-20,8337.000000,4764.000000,-17564.000000,9536.000000,3372.000000,1568.000000,-16714.000000,-7119.000000,-20261.000000,14136.000000,5740.000000
663,Portfolio Equity Liabilities,Thailand,USD mn,2026-01-26,-1928.646505,-1455.774645,-1531.458275,-741.669460,-400.681064,-831.121553,1315.281433,-3478.960505,-306.260984,655.999050,117.911222


In [36]:
asset_item =['FDI Assets', 'FDI Equity Assets', 'FDI Debt Assets', 'Portfolio Assets', 'Portfolio Equity Assets', 'Portfolio Debt Assets', 'Financial Derivative Assets', 'Other Investment Assets', 'OI Equity Assets', 'Currency and Deposits Assets', 'Loans Assets', 'Insurance and Pension Assets', 'Trade Credits and Advances Assets', 'OI Others Assets', 'Official Reserve Assets']
liabi_item =['FDI Liabilities', 'FDI Equity Liabilities', 'FDI Debt Liabilities', 'Portfolio Liabilities', 'Portfolio Equity Liabilities', 'Portfolio Debt Liabilities', 'Financial Derivative Liabilities', 'Other Investment Liabilities', 'OI Equity Liabilities', 'Currency and Deposits Liabilities', 'Loans Liabilities', 'Insurance and Pension Liabilities', 'Trade Credits and Advances Liabilities', 'OI Others Liabilities', 'SDR Liabilities']

all1 = set(asset_item + liabi_item)
all1

{'Currency and Deposits Assets',
 'Currency and Deposits Liabilities',
 'FDI Assets',
 'FDI Debt Assets',
 'FDI Debt Liabilities',
 'FDI Equity Assets',
 'FDI Equity Liabilities',
 'FDI Liabilities',
 'Financial Derivative Assets',
 'Financial Derivative Liabilities',
 'Insurance and Pension Assets',
 'Insurance and Pension Liabilities',
 'Loans Assets',
 'Loans Liabilities',
 'OI Equity Assets',
 'OI Equity Liabilities',
 'OI Others Assets',
 'OI Others Liabilities',
 'Official Reserve Assets',
 'Other Investment Assets',
 'Other Investment Liabilities',
 'Portfolio Assets',
 'Portfolio Debt Assets',
 'Portfolio Debt Liabilities',
 'Portfolio Equity Assets',
 'Portfolio Equity Liabilities',
 'Portfolio Liabilities',
 'SDR Liabilities',
 'Trade Credits and Advances Assets',
 'Trade Credits and Advances Liabilities'}

In [70]:

# import libraries

import sdmx

 

# retrieve data

IMF_DATA = sdmx.Client('IMF_DATA')

data_msg = IMF_DATA.data('CPI', params={'startPeriod': 2018})

 

# convert to pandas

cpi_df = sdmx.to_pandas(data_msg)

print(cpi_df.head())

xml.Reader got no structure=… argument for StructureSpecificData


TIME_PERIOD  INDEX_TYPE  COICOP_1999  TYPE_OF_TRANSFORMATION  COUNTRY  FREQUENCY  COMMON_REFERENCE_PERIOD  OVERLAP  SCALE  ACCESS_SHARING_LEVEL  SECURITY_CLASSIFICATION  METHODOLOGY  IFS_FLAG
2018         CPI         CP01         IX                      ABW      A          2019M6                   OL       0      PUBLIC_OPEN           PUB                                               139.228833
2019         CPI         CP01         IX                      ABW      A          2019M6                   OL       0      PUBLIC_OPEN           PUB                                               157.077250
2018-M01     CPI         CP01         IX                      ABW      M          2019M6                   OL       0      PUBLIC_OPEN           PUB                                               128.733000
2018-M02     CPI         CP01         IX                      ABW      M          2019M6                   OL       0      PUBLIC_OPEN           PUB                                          

In [53]:
len(len_date)

37505449